# Семинар 07. Хеш-таблицы

Пять задач про то, как словарь и множество отвечают на вопрос «уже встречали?». В каждой сначала
решаем, что считать ключом и какую операцию нужно сделать быстрой, и только потом пишем код.

## Цели

- выбирать между списком, словарём и множеством по нужной операции;
- собирать свод по составному ключу, помня правило «последняя запись побеждает» и порядок первого появления;
- отвечать на вопросы «ровно в одном», «ровно в двух» операциями над множествами и не путать их с симметрической разностью;
- поддерживать два согласованных индекса при вставке, перемещении и удалении;
- сводить вопрос про непрерывный отрезок к префиксным суммам и словарю.

## Перед началом

Откройте [лекцию](lecture.ipynb). Задания идут от простых к сложным и разбираются подряд;
задание 6 — дополнительное, для тех, кто закончил раньше. Для каждой задачи сначала ответьте
себе на три вопроса: что здесь ключ, что считается повтором и какую операцию нужно сделать
быстрой.

Всё используемое входит в стандартную библиотеку и работает на Python 3.10 и новее, поэтому
ноутбук открывается и локально, и в Colab.

Замените `raise NotImplementedError` своим кодом. Сначала запустите ячейку с определением
функции, затем ячейку проверок под ней. До появления решения проверки намеренно не проходят.
Добавьте к каждому решению собственный пример.

Внутри функций не используйте `input` и `print` и не меняйте входные данные. Все входные данные
соответствуют условиям. Оценивая работу `dict` и `set`, считаем хеширование и сравнение коротких
ключей операциями постоянной стоимости, а линейные оценки — ожидаемыми.

## Разминка. Что напечатает

Три фрагмента, каждый про отдельное свойство хеш-контейнеров. Сначала запишите прогноз, затем
выполните ячейку и сверьтесь. Разбор — под спойлером после всех фрагментов.

In [ ]:
# Фрагмент 1
status = {"new": 3, "done": 8, "error": 1}

status["done"] = 9
print("после замены значения:", list(status))

del status["done"]
status["done"] = 10
print("после удаления и вставки:", list(status))

In [ ]:
# Фрагмент 2
first = {"anna", "boris", "vera"}
second = {"boris", "vera", "gleb"}
third = {"vera", "gleb", "dasha"}

print("симметрическая разность:", sorted(first ^ second ^ third))

visits = {name: (name in first) + (name in second) + (name in third)
          for name in first | second | third}
print("ровно в двух:", sorted(name for name, times in visits.items() if times == 2))

In [ ]:
# Фрагмент 3
prices = {("milk", 2026): 70}
print(prices[("milk", 2026)])

print({(1, 2): "первое", (1.0, 2): "второе"})

try:
    shelf = {["milk", 2026]: 70}
except TypeError as error:
    print(type(error).__name__, "-", error)

<details>
<summary>Разбор</summary>

**Фрагмент 1.** `['new', 'done', 'error']`, затем `['new', 'error', 'done']`. Словарь хранит
порядок вставки. Замена значения по существующему ключу — не вставка: место ключа не меняется.
А вот удаление с последующей вставкой создаёт ключ заново, и он уходит в конец. Если в условии
сказано «в порядке первого появления», замена значения этот порядок не портит, но удалять ключ
по пути нельзя.

**Фрагмент 2.** `['anna', 'dasha', 'vera']`, затем `['boris', 'gleb']`. Симметрическая разность
трёх множеств оставляет тех, кто встретился нечётное число раз, — то есть и одиночек, и тех, кто
есть во всех трёх. `vera` попала в ответ `^`, хотя была везде. Вопросы вида «ровно в одном» и
«ровно в двух» решаются подсчётом вхождений, а не комбинацией `|`, `&` и `^`.

**Фрагмент 3.** `70`, затем `{(1, 2): 'второе'}`, затем `TypeError - unhashable type: 'list'`.
Кортеж годится в ключи, пока хешируются все его элементы, — так получается составной ключ.
Список не хешируется: его можно изменить, и ключ «уехал» бы из своей корзины. А `(1, 2)` и
`(1.0, 2)` — один и тот же ключ: `1 == 1.0`, хеши равны, второе значение просто заменило
первое.

</details>

### Задание 1. Кто улучшил результат

В журнале защит записаны попытки: `(login, homework, score)` — логин, номер домашнего задания и
балл. Записи идут в порядке проведения защит, один и тот же студент может защищать одно и то же
задание несколько раз.

Напишите `improved(records)`: верните список пар `(login, homework)`, у которых **последний** балл
строго больше **первого**. Пары идут в порядке первого появления в журнале.

- Если пара встречается в журнале один раз, улучшения нет.
- Разные задания одного студента — разные пары.
- Повторная защита с тем же баллом улучшением не считается.

До 100 000 записей; логин — строка, номер задания и балл — целые. Ожидаемое время `O(n)`.
Входной список не меняйте.

Пример: для журнала

```python
[("rita", 1, 4), ("ivan", 1, 7), ("rita", 1, 9), ("ivan", 1, 7), ("rita", 2, 3)]
```

ответ — `[("rita", 1)]`: у Риты по первому заданию балл вырос с 4 до 9, у Ивана остался прежним,
а второе задание Риты защищалось один раз.

In [ ]:
def improved(records: list[tuple[str, int, int]]) -> list[tuple[str, int]]:
    raise NotImplementedError

In [ ]:
assert improved([]) == []
assert improved([("rita", 1, 4)]) == []
assert improved([("rita", 1, 4), ("rita", 1, 9)]) == [("rita", 1)]
assert improved([("rita", 1, 9), ("rita", 1, 4)]) == []
assert improved([("rita", 1, 4), ("rita", 1, 4)]) == []
assert improved([
    ("rita", 1, 4), ("ivan", 1, 7), ("rita", 1, 9), ("ivan", 1, 7), ("rita", 2, 3),
]) == [("rita", 1)]

# Разные задания одного студента не смешиваются.
assert improved([("rita", 1, 4), ("rita", 2, 1), ("rita", 2, 8)]) == [("rita", 2)]

# Порядок — по первому появлению пары, а не по моменту улучшения.
assert improved([
    ("anna", 1, 1), ("boris", 1, 1), ("boris", 1, 5), ("anna", 1, 5),
]) == [("anna", 1), ("boris", 1)]

# Сравниваются первый и последний баллы, промежуточные не важны.
assert improved([("anna", 1, 5), ("anna", 1, 9), ("anna", 1, 5)]) == []

journal = [("rita", 1, 4), ("rita", 1, 9)]
before = journal.copy()
result = improved(journal)
assert result == [("rita", 1)]
assert journal == before

### Задание 2. Пришёл только однажды

Для трёх консультаций собрали списки участников. Внутри одного списка имя могло отметиться
несколько раз — это один и тот же человек.

Напишите `only_once(first, second, third)`: верните **множество** имён тех, кто есть ровно в
одном из трёх списков. Повторные отметки внутри списка на ответ не влияют.

Списки могут быть пустыми, суммарно до 300 000 имён. Ожидаемое время — линейное от суммарной
длины. Входные списки не меняйте.

Пример: `["anna", "boris", "anna"]`, `["boris", "vera"]`, `["vera", "gleb"]` дают
`{"anna", "gleb"}` — Борис и Вера были на двух консультациях каждый.

Перед кодом проверьте на разминке, годится ли здесь `first ^ second ^ third`.

In [ ]:
def only_once(first: list[str], second: list[str], third: list[str]) -> set[str]:
    raise NotImplementedError

In [ ]:
assert only_once([], [], []) == set()
assert only_once(["anna"], [], []) == {"anna"}
assert only_once(["anna", "anna"], [], []) == {"anna"}
assert only_once(["anna", "boris", "anna"], ["boris", "vera"], ["vera", "gleb"]) == {
    "anna", "gleb",
}
assert only_once(["anna"], ["anna"], []) == set()

# Тот, кто был на всех трёх, в ответ не попадает — хотя симметрическая разность его оставляет.
assert only_once(["vera"], ["vera"], ["vera"]) == set()
assert {"vera"} ^ {"vera"} ^ {"vera"} == {"vera"}

assert only_once(["a", "b"], ["b", "c"], ["c", "a"]) == set()
assert only_once(["a"], ["b"], ["c"]) == {"a", "b", "c"}

visitors = ["anna", "anna"]
before = visitors.copy()
assert only_once(visitors, ["boris"], []) == {"anna", "boris"}
assert visitors == before
assert type(only_once(["anna"], [], [])) is set

### Задание 3. Сколько непрерывных серий

Сервис записывает номера дней, когда он работал без сбоев. Записи приходят в произвольном
порядке, один и тот же день может быть записан несколько раз.

Напишите `count_runs(days)`: верните число **максимальных** отрезков подряд идущих дней. Отрезок
максимален, если его нельзя продлить ни влево, ни вправо: соседних дней в журнале нет.

До 200 000 записей, номера дней — целые. Нужно ожидаемое `O(n)`, поэтому сортировка не подойдёт.
Входной список не меняйте.

Пример: в `[7, 3, 5, 4, 4, 10, 2]` различные дни — это `2, 3, 4, 5, 7, 10`. Они образуют три
отрезка: `2–5`, `7` и `10`, поэтому ответ `3`.

In [ ]:
def count_runs(days: list[int]) -> int:
    raise NotImplementedError

In [ ]:
assert count_runs([]) == 0
assert count_runs([5]) == 1
assert count_runs([5, 5, 5]) == 1
assert count_runs([7, 3, 5, 4, 4, 10, 2]) == 3
assert count_runs([1, 2, 3]) == 1
assert count_runs([3, 2, 1]) == 1
assert count_runs([1, 3, 5]) == 3
assert count_runs([10, 11, 1, 2]) == 2
assert count_runs([0, 1000000000]) == 2
assert count_runs([-3, -2, 4]) == 2

journal = [7, 3, 5, 4]
before = journal.copy()
assert count_runs(journal) == 2
assert journal == before

### Задание 4. Каталог полок

В библиотеке ведут каталог: где лежит каждая книга. Команды приходят по порядку:

| Команда | Что делает |
| --- | --- |
| `("PUT", book, shelf)` | положить книгу на полку; если книга уже где-то лежала, переставить её |
| `("REMOVE", book)` | убрать книгу из каталога; если такой книги нет, ничего не менять |
| `("COUNT", shelf)` | сколько книг сейчас на этой полке |

Напишите `shelf_report(commands)`: верните список ответов на команды `COUNT` в порядке их
появления. Для пустой полки ответ `0`.

До 200 000 команд; названия книг и номера полок — строки и целые числа. На каждую команду должно
уходить ожидаемое `O(1)`, поэтому пересчитывать полку при каждом `COUNT` нельзя. Входной список
не меняйте.

Пример: `[("PUT", "ulysses", 1), ("PUT", "hamlet", 1), ("COUNT", 1), ("PUT", "hamlet", 2),
("COUNT", 1), ("COUNT", 2), ("REMOVE", "ulysses"), ("COUNT", 1)]` даёт `[2, 1, 1, 0]`.

In [ ]:
def shelf_report(commands: list[tuple]) -> list[int]:
    raise NotImplementedError

In [ ]:
assert shelf_report([]) == []
assert shelf_report([("COUNT", 1)]) == [0]
assert shelf_report([("PUT", "ulysses", 1), ("COUNT", 1)]) == [1]

assert shelf_report([
    ("PUT", "ulysses", 1), ("PUT", "hamlet", 1), ("COUNT", 1),
    ("PUT", "hamlet", 2), ("COUNT", 1), ("COUNT", 2),
    ("REMOVE", "ulysses"), ("COUNT", 1),
]) == [2, 1, 1, 0]

# Повторный PUT на ту же полку не удваивает счётчик.
assert shelf_report([("PUT", "a", 1), ("PUT", "a", 1), ("COUNT", 1)]) == [1]

# Удаление того, чего нет, ничего не меняет.
assert shelf_report([("REMOVE", "a"), ("COUNT", 1)]) == [0]
assert shelf_report([("PUT", "a", 1), ("REMOVE", "a"), ("REMOVE", "a"), ("COUNT", 1)]) == [0]

# Переезд уменьшает старую полку и увеличивает новую.
assert shelf_report([
    ("PUT", "a", 1), ("PUT", "a", 2), ("COUNT", 1), ("COUNT", 2),
]) == [0, 1]

assert shelf_report([("PUT", "a", 7), ("PUT", "b", 7), ("REMOVE", "b"), ("COUNT", 7)]) == [1]

commands = [("PUT", "a", 1), ("COUNT", 1)]
before = commands.copy()
assert shelf_report(commands) == [1]
assert commands == before

### Задание 5. Кратчайший отрезок с нужной суммой

Кассовый журнал хранит операции в порядке проведения: положительная сумма — поступление,
отрицательная — возврат.

Напишите `shortest_segment(values, target)`: верните длину самого короткого непрерывного отрезка
операций, сумма которого равна `target`. Если такого отрезка нет, верните `0`.

Отрезок состоит хотя бы из одной операции. Суммы могут быть отрицательными, поэтому расширять и
сжимать окно по знаку суммы не получится.

До 200 000 операций. Нужно ожидаемое `O(n)`: перебор всех пар границ слишком дорог. Входной
список не меняйте.

Пример: в `[1, -1, 1, 0, -1, 2]` при `target = 1` подходят несколько отрезков, самый короткий —
одна операция `1`, поэтому ответ `1`. При `target = 0` самый короткий — `[0]`, тоже длиной `1`.

После решения объясните, какая величина служит ключом словаря и что хранится в значении.

In [ ]:
def shortest_segment(values: list[int], target: int) -> int:
    raise NotImplementedError

In [ ]:
assert shortest_segment([], 0) == 0
assert shortest_segment([5], 5) == 1
assert shortest_segment([5], 4) == 0
assert shortest_segment([1, -1, 1, 0, -1, 2], 1) == 1
assert shortest_segment([1, -1, 1, 0, -1, 2], 0) == 1
assert shortest_segment([1, 2, 3], 6) == 3
assert shortest_segment([1, 2, 3], 5) == 2
assert shortest_segment([1, 2, 3], 7) == 0

# Берётся кратчайший из нескольких подходящих.
assert shortest_segment([3, 1, 2, 3], 3) == 1
assert shortest_segment([2, 1, 2], 3) == 2

# Отрицательные суммы и нулевой отрезок.
assert shortest_segment([-1, -2, -3], -5) == 2
assert shortest_segment([4, -4, 7], 0) == 2
assert shortest_segment([1, 2, -3, 4], 0) == 3

journal = [1, -1, 2]
before = journal.copy()
assert shortest_segment(journal, 2) == 1
assert journal == before

## Дополнительно. Размещение без коллизий

Задание для тех, кто закончил раньше.

В лекции коллизии разрешались цепочками. Теперь все ключи известны заранее, новых не будет, и
хочется таблицу, в которой **ни одной коллизии нет**. Номер ячейки считается единственным
способом: `h(key) = key % capacity`.

Напишите `smallest_capacity(keys)`: верните наименьшее положительное число ячеек, при котором все
ключи попадают в разные ячейки. Для пустого набора ответ `1`.

Ключи — различные целые числа от 0 до 20 000, не более 200 штук. Входной список не меняйте.

Пример: для `[0, 3, 4]` размеры 3 и 4 не подходят, потому что `3 % 3 == 0 % 3` и
`4 % 4 == 0 % 4`, а размер 5 подходит — остатки `0`, `3` и `4` различны.

Ёмкость не обязана равняться числу ключей. Объясните, почему поиск обязательно завершится и
почему ёмкость меньше числа ключей рассматривать бессмысленно.

In [ ]:
def smallest_capacity(keys: list[int]) -> int:
    raise NotImplementedError

In [ ]:
assert smallest_capacity([]) == 1
assert smallest_capacity([7]) == 1
assert smallest_capacity([0, 1]) == 2
assert smallest_capacity([0, 3, 4]) == 5
assert smallest_capacity([5, 10]) == 2
assert smallest_capacity([0, 2, 4]) == 3
assert smallest_capacity([1, 2, 3]) == 3
assert smallest_capacity([0, 1, 2, 3]) == 4
assert smallest_capacity([10, 20, 30]) == 3

keys = [0, 3, 4]
before = keys.copy()
assert smallest_capacity(keys) == 5
assert keys == before

## Итоги

- Ключ решает всё: по паре `(логин, задание)` журнал распадается на независимые истории, а по
  одному логину — склеивается.
- Словарь хранит порядок вставки, и замена значения его не меняет; удаление с повторной вставкой
  меняет.
- Вопросы «ровно в одном» и «ровно в двух» решаются подсчётом вхождений, а не комбинацией
  операций над множествами.
- Два индекса стоит держать согласованными сразу: при перемещении и удалении старый счётчик
  нужно уменьшить в том же месте кода.
- Вопрос о непрерывном отрезке часто сводится к префиксным суммам, а словарь превращает поиск
  нужного префикса в одну операцию.

Для каждого решения назовите три вещи: что здесь ключ, что считается повтором и какую операцию
вы сделали быстрой.

[Домашка](https://contest.yandex.ru/contest/CONTEST_ID/enter/?retPage=).